In [7]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [8]:
# loading the dataset
filepath = os.path.join(os.path.abspath(".."), "data", "course_lead_scoring_2026.csv")
df = pd.read_csv(filepath)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [9]:
# inspect the dtypes, missing values, etc.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [10]:
# describe the numerical quantaties
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


In [19]:
# number of unique values
nums = df.drop(columns=["converted"]).select_dtypes(include=[np.number]).columns
cat = df.columns[df.dtypes == "str"]
print(nums)
for col in cat:
    print(f"{col}: {df[col].nunique()}")

Index(['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score'],
      dtype='str')
lead_source: 5
industry: 6
employment_status: 4
location: 5


In [20]:
# imbalance check
df.converted.value_counts(normalize=True)

converted
1    0.5766
0    0.4234
Name: proportion, dtype: float64

In [21]:
# filling missing values
df[nums] = df[nums].fillna(df[nums].median())
df[cat] = df[cat].fillna("NA")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               5000 non-null   str    
 1   industry                  5000 non-null   str    
 2   employment_status         5000 non-null   str    
 3   location                  5000 non-null   str    
 4   annual_income             5000 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                5000 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [22]:
# feature, target splitting
y = df.converted.values
X = df.drop(columns=["converted"])
X.shape, y.shape

((5000, 8), (5000,))

In [23]:
# train, validation, and test split
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=11, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=11, stratify=y_train_full)
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((3000, 8), (3000,), (1000, 8), (1000,), (1000, 8), (1000,))

In [24]:
# one hot encoding 
encoder = OneHotEncoder(handle_unknown="ignore")
X_train_transformed = encoder.fit_transform(X_train[cat])
columns = encoder.get_feature_names_out(cat)
X_train_encoded = pd.DataFrame(
    X_train_transformed.toarray(),
    columns=columns,
    index=X_train.index,
)
X_train = pd.concat(
    [X_train[nums], X_train_encoded], axis=1
)
X_train = X_train.reset_index(drop=True)

X_val_transformed = encoder.transform(X_val[cat])
X_val_encoded = pd.DataFrame(
    X_val_transformed.toarray(),
    columns=columns,
    index=X_val.index,
)
X_val = pd.concat(
    [X_val[nums], X_val_encoded], axis=1
)
X_val = X_val.reset_index(drop=True)

In [25]:
# extra 4 for NA
X_train.shape, X_train_transformed.shape

((3000, 28), (3000, 24))

In [26]:
encoder.get_feature_names_out(cat)

array(['lead_source_NA', 'lead_source_events',
       'lead_source_organic_search', 'lead_source_paid_ads',
       'lead_source_referral', 'lead_source_social_media', 'industry_NA',
       'industry_education', 'industry_finance', 'industry_healthcare',
       'industry_manufacturing', 'industry_retail', 'industry_technology',
       'employment_status_NA', 'employment_status_employed',
       'employment_status_self_employed', 'employment_status_student',
       'employment_status_unemployed', 'location_NA', 'location_africa',
       'location_asia', 'location_europe', 'location_north_america',
       'location_south_america'], dtype=object)

In [27]:
model = LogisticRegression(solver="liblinear", C=1.0)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]

In [28]:
accuracy_score(y_val, y_pred >= 0.5)

0.682

In [29]:
roc_auc_score(y_val, y_pred >= 0.5)

0.6452507671947917

In [30]:
print(classification_report(y_val, y_pred >= 0.5))

              precision    recall  f1-score   support

           0       0.72      0.41      0.52       423
           1       0.67      0.88      0.76       577

    accuracy                           0.68      1000
   macro avg       0.69      0.65      0.64      1000
weighted avg       0.69      0.68      0.66      1000



In [31]:
print(confusion_matrix(y_val, y_pred >= 0.5))

[[172 251]
 [ 67 510]]


In [32]:
# --- Combine ---
# filling missing values
X[cat] = X[cat].fillna("NA")
X[nums] = X[nums].fillna(X[nums].median())
# train, validation, and test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=11, stratify=y)
# one hot encoding 
encoder = OneHotEncoder(handle_unknown="ignore")
X_train_transformed = encoder.fit_transform(X_train[cat])
columns = encoder.get_feature_names_out(cat)
X_train_encoded = pd.DataFrame(
    X_train_transformed.toarray(),
    columns=columns,
    index=X_train.index,
)
X_train = pd.concat(
    [X_train[nums], X_train_encoded], axis=1
)
X_train = X_train.reset_index(drop=True)

X_test_transformed = encoder.transform(X_test[cat])
X_test_encoded = pd.DataFrame(
    X_test_transformed.toarray(),
    columns=columns,
    index=X_test.index,
)
X_test = pd.concat(
    [X_test[nums], X_test_encoded], axis=1
)
X_test = X_test.reset_index(drop=True)

In [33]:
model = LogisticRegression(solver="liblinear", C=1.0)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]

In [34]:
accuracy_score(y_test, y_pred >= 0.5)

0.667

In [35]:
roc_auc_score(y_test, y_pred >= 0.5)

0.6322525002970447

In [36]:
print(classification_report(y_test, y_pred >= 0.5))

              precision    recall  f1-score   support

           0       0.68      0.41      0.51       423
           1       0.66      0.86      0.75       577

    accuracy                           0.67      1000
   macro avg       0.67      0.63      0.63      1000
weighted avg       0.67      0.67      0.65      1000



In [37]:
print(confusion_matrix(y_test, y_pred >= 0.5))

[[172 251]
 [ 82 495]]


In [38]:
# --- pipeline ---
df = pd.read_csv(filepath)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [39]:
# --- Numerical pipeline ---
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# --- Categorical pipeline ---
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="NA")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# --- Combine ---
preprocessor = ColumnTransformer(transformers=[
    ("nums", numerical_transformer, nums),
    ("cat", categorical_transformer, cat),
])

# --- Model ---
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(solver="liblinear", C=1.0))
])

In [40]:
# train, test, split
y = df.converted.values
X = df.drop(columns=["converted"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=11, stratify=y)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((4000, 8), (4000,), (1000, 8), (1000,))

In [41]:
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]
print(f"acc: {accuracy_score(y_test, y_pred >= 0.5)}")
print(f"roc_auc_score: {roc_auc_score(y_test, y_pred >= 0.5)}")
print(f"classification_report: {classification_report(y_test, y_pred >= 0.5)}")
print(f"confusion_matrix: {confusion_matrix(y_test, y_pred >= 0.5)}")

acc: 0.717
roc_auc_score: 0.7046043979006109
classification_report:               precision    recall  f1-score   support

           0       0.68      0.62      0.65       423
           1       0.74      0.79      0.76       577

    accuracy                           0.72      1000
   macro avg       0.71      0.70      0.71      1000
weighted avg       0.71      0.72      0.72      1000

confusion_matrix: [[264 159]
 [124 453]]
